In [20]:
import sys
from pathlib import Path
import torch

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [61]:
import Python.simfun as sim


X1, y1, beta1, sim_info1 = sim.simfun1(
    sim="simple",
    n=160,
    p=100,
    seed=123,
    n_active=10,
    sigma2=1,
    device=device,
    dtype=torch.float32,
)


X2, y2, beta2, sim_info2 = sim.simfun1(
    sim="simple",
    n=100,
    p=500,
    seed=123,
    n_active=10,
    sigma2=1,
    device=device,
    dtype=torch.float32,
)


In [ ]:
%load_ext autoreload
%autoreload 2

import Python.config as cfg
import Python.framework as fw
import Python.model as md


split_cfg = cfg.SplitConfig(
    train_frac=0.6,
    val_frac=0.2,
    test_frac=0.2,
    seed=123,
)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [60]:
schedule_cfg1 = cfg.StagewiseAnnealConfig(
    tau_start=0.3,
    tau_end=0.1,
    warmup_epochs=300,
    n_anneal_stages=20,
    min_stage_epochs=100,
    max_stage_epochs=400,
    base_lr=5e-5,
    stage_lr_decay=0.7,
    min_lr_scale = 0.2,
    eval_every=25,
    print_every=100,
    diag_R_train=256,
    diag_R_final=4000,
    support_threshold=0.05,
)

out1 = fw.simflow_stagewise(
    X=X1,
    y=y1,
    beta_true=beta1,
    sim_info=sim_info1,
    build_flow_vi=md.build_flow_vi,
    family="gaussian",
    seed=123,
    K_q=32,
    K_g=16,
    schedule_cfg=schedule_cfg1,
    split_cfg=split_cfg,
    save_cfg=cfg.SaveConfig(output_dir=None),
)

===== Simulation info =====
Simulation summary:
  sim                   : simple
  n                     : 160
  p                     : 100
  n_active              : 10
  sigma2                : 1.0000
  sigma                 : 1.0000
  signal_var            : 10.7965
  outcome_var           : 12.1076
  snr_actual            : 10.7965
  beta_low              : 0.3000
  beta_high             : 2.0000
  center_y              : True

Active indices, zero-based:
  [13, 20, 32, 44, 49, 56, 64, 79, 84, 94]

Active variables sorted by |beta_true|:
  rank     j    j1    beta_true       |beta|
     1    94    95       1.8760       1.8760
     2    64    65      -1.8558       1.8558
     3    32    33       1.5538       1.5538
     4    84    85      -0.8610       0.8610
     5    20    21      -0.8007       0.8007
     6    13    14       0.7566       0.7566
     7    49    50      -0.7473       0.7473
     8    44    45      -0.5824       0.5824
     9    56    57       0.5678       0.5678
  

In [62]:
schedule_cfg2 = cfg.StagewiseAnnealConfig(
    tau_start=0.5,
    tau_end=0.1,
    warmup_epochs=300,
    n_anneal_stages=20,
    min_stage_epochs=100,
    max_stage_epochs=400,
    base_lr=2e-5,
    stage_lr_decay=0.9,
    min_lr_scale = 0.5,
    eval_every=25,
    print_every=100,
    diag_R_train=256,
    diag_R_final=4000,
    support_threshold=0.05,
)

out2 = fw.simflow_stagewise(
    X=X1,
    y=y1,
    beta_true=beta1,
    sim_info=sim_info1,
    build_flow_vi=md.build_flow_vi,
    family="gaussian",
    seed=123,
    K_q=32,
    K_g=16,
    schedule_cfg=schedule_cfg2,
    split_cfg=split_cfg,
    save_cfg=cfg.SaveConfig(output_dir=None),
)

===== Simulation info =====
Simulation summary:
  sim                   : simple
  n                     : 160
  p                     : 100
  n_active              : 10
  sigma2                : 1.0000
  sigma                 : 1.0000
  signal_var            : 10.7965
  outcome_var           : 12.1076
  snr_actual            : 10.7965
  beta_low              : 0.3000
  beta_high             : 2.0000
  center_y              : True

Active indices, zero-based:
  [13, 20, 32, 44, 49, 56, 64, 79, 84, 94]

Active variables sorted by |beta_true|:
  rank     j    j1    beta_true       |beta|
     1    94    95       1.8760       1.8760
     2    64    65      -1.8558       1.8558
     3    32    33       1.5538       1.5538
     4    84    85      -0.8610       0.8610
     5    20    21      -0.8007       0.8007
     6    13    14       0.7566       0.7566
     7    49    50      -0.7473       0.7473
     8    44    45      -0.5824       0.5824
     9    56    57       0.5678       0.5678
  

In [49]:
import numpy as np

X_np = X1.detach().cpu().numpy()
y_np = y1.detach().cpu().numpy()
beta_np = beta1.detach().cpu().numpy()

active = np.where(np.abs(beta_np) > 1e-12)[0]

marginal = np.abs(X_np.T @ y_np) / X_np.shape[0]
marg_rank = np.argsort(-marginal)
marg_pos = {j: r + 1 for r, j in enumerate(marg_rank)}

for j in active:
    print(
        "j =", j,
        "beta =", beta_np[j],
        "abs_beta =", abs(beta_np[j]),
        "marginal =", marginal[j],
        "marginal_rank =", marg_pos[j],
    )

j = 4 beta = -0.7575302 abs_beta = 0.7575302 marginal = 0.7974108 marginal_rank = 15
j = 8 beta = -0.4434043 abs_beta = 0.4434043 marginal = 0.8679196 marginal_rank = 13
j = 14 beta = 1.1809521 abs_beta = 1.1809521 marginal = 1.3310775 marginal_rank = 7
j = 23 beta = 1.497907 abs_beta = 1.497907 marginal = 1.9730439 marginal_rank = 1
j = 121 beta = 1.2285146 abs_beta = 1.2285146 marginal = 1.4473299 marginal_rank = 6
j = 122 beta = 0.898026 abs_beta = 0.898026 marginal = 1.5403488 marginal_rank = 5
j = 134 beta = -1.3559818 abs_beta = 1.3559818 marginal = 1.6588207 marginal_rank = 4
j = 165 beta = 1.5775198 abs_beta = 1.5775198 marginal = 1.8066832 marginal_rank = 2
j = 174 beta = -1.0678335 abs_beta = 1.0678335 marginal = 1.7205337 marginal_rank = 3
j = 177 beta = 0.98963505 abs_beta = 0.98963505 marginal = 0.20547995 marginal_rank = 125


In [50]:
selected = np.array([14, 23, 121, 122, 134, 165, 174])

coef_sel = np.linalg.lstsq(X_np[:, selected], y_np, rcond=None)[0]
resid = y_np - X_np[:, selected] @ coef_sel

resid_score = np.abs(X_np.T @ resid) / X_np.shape[0]
resid_rank = np.argsort(-resid_score)
resid_pos = {j: r + 1 for r, j in enumerate(resid_rank)}

for j in active:
    print(
        "j =", j,
        "beta =", beta_np[j],
        "abs_beta =", abs(beta_np[j]),
        "resid_score =", resid_score[j],
        "resid_rank =", resid_pos[j],
    )

j = 4 beta = -0.7575302 abs_beta = 0.7575302 resid_score = 0.5019819 resid_rank = 2
j = 8 beta = -0.4434043 abs_beta = 0.4434043 resid_score = 0.4494448 resid_rank = 3
j = 14 beta = 1.1809521 abs_beta = 1.1809521 resid_score = 5.1227453e-08 resid_rank = 196
j = 23 beta = 1.497907 abs_beta = 1.497907 resid_score = 7.3223987e-09 resid_rank = 199
j = 121 beta = 1.2285146 abs_beta = 1.2285146 resid_score = 2.1979409e-08 resid_rank = 197
j = 122 beta = 0.898026 abs_beta = 0.898026 resid_score = 3.5288622e-10 resid_rank = 200
j = 134 beta = -1.3559818 abs_beta = 1.3559818 resid_score = 7.357565e-08 resid_rank = 194
j = 165 beta = 1.5775198 abs_beta = 1.5775198 resid_score = 5.9547553e-08 resid_rank = 195
j = 174 beta = -1.0678335 abs_beta = 1.0678335 resid_score = 7.5710185e-09 resid_rank = 198
j = 177 beta = 0.98963505 abs_beta = 0.98963505 resid_score = 0.66271406 resid_rank = 1
